In [ ]:
!pip install tsai 
import sys

# put the repo first on sys.path so this geoxts is the one imported
if '/Users/fokia/CRP_SLB' not in sys.path:
    sys.path.insert(0, '/Users/fokia/CRP_SLB')

from geoxts.benchmark_data import *


In [ ]:
!pip install parquet
import parquet
import pandas as pd 

import pandas as pd 

df = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/north_sea/data/npd_welllogs_combined.parquet')

# wells that have tops
tops_raw = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/north_sea/notebooks/Data_NPD/npd_tops.parquet')
tops_raw = tops_raw.rename(columns={'wellbore_name': 'well_name'})




In [ ]:
df

In [ ]:
tops_raw = tops_raw.reset_index()


In [ ]:
tops_raw

In [ ]:
tops_raw['well_name'].unique()

In [ ]:
wells_with_tops = tops_raw['well_name'].unique()
# keep only wells with formation tops
df_before = df['well_name'].nunique() if 'well_name' in df.columns else df.index.nunique()
df = df[df['well_name'].isin(wells_with_tops)]
print(f"Wells before tops filter : {df_before}")
print(f"Wells after tops filter  : {df['well_name'].nunique()}")
print(f"Wells dropped            : {df_before - df['well_name'].nunique()}")
print(f"Rows remaining           : {len(df):,}")

In [ ]:
df.isna().sum()

In [ ]:
is_nan = df['RHOB_CLEAN'].isna()

# length of each run of consecutive NaNs
gap_sizes = is_nan.groupby((~is_nan).cumsum()).sum()
large_gaps = gap_sizes[gap_sizes > 100]
print(f"Largest consecutive missing gap: {gap_sizes.max()} rows")
print(f"Smallest gap: {gap_sizes.max()} rows")
print(f"Average missing gap size: {gap_sizes[gap_sizes > 0].mean():.1f} rows")


In [ ]:
temp_df = df[['well_name']].copy()
temp_df['is_missing'] = df['RHOB_CLEAN'].isna()
temp_df['block'] = (~temp_df['is_missing']).cumsum()

missing_rows = temp_df[temp_df['is_missing']]

# rows per missing block, per well
gap_sizes = missing_rows.groupby(['well_name', 'block']).size().reset_index(name='gap_length')
mean_gap = gap_sizes['gap_length'].mean()

# blocks longer than the mean gap
massive_gaps = gap_sizes[gap_sizes['gap_length'] > float(mean_gap)]
small_gaps = gap_sizes[gap_sizes['gap_length'] <= float(mean_gap)]

affected_wells = massive_gaps['well_name'].unique()
n_affected_wells = small_gaps['well_name'].unique()
print(f"Number of wells with more than {mean_gap} row gaps: {len(affected_wells)}")
print(f"Number of wells with less than {mean_gap} row gaps: {len(n_affected_wells)}")


In [ ]:
# step 1: linear interpolation per well, fills runs of up to 100 missing rows (~15 m)
print(f"Original missing RHOB values        : {df['RHOB_CLEAN'].isna().sum():,}")
df['RHOB_CLEAN'] = df.groupby('well_name')['RHOB_CLEAN'].transform(
    lambda well: well.interpolate(method='linear', limit=100, limit_direction='both')
)
print(f"Missing values after patching glitches: {df['RHOB_CLEAN'].isna().sum():,}\n")

print(f"Original missing GR_CLEAN values        : {df['GR_CLEAN'].isna().sum():,}")
df['GR_CLEAN'] = df.groupby('well_name')['GR_CLEAN'].transform(
    lambda well: well.interpolate(method='linear', limit=100, limit_direction='both')
)
print(f"Missing values after patching glitches: {df['GR_CLEAN'].isna().sum():,}\n")

print(f"Original missing DEEPRES_CLEAN values        : {df['DEEPRES_CLEAN'].isna().sum():,}")
df['DEEPRES_CLEAN'] = df.groupby('well_name')['DEEPRES_CLEAN'].transform(
    lambda well: well.interpolate(method='linear', limit=100, limit_direction='both')
)
print(f"Missing values after patching glitches: {df['DEEPRES_CLEAN'].isna().sum():,}\n")



In [ ]:
# step 2: gaps still left after interpolation
is_nan = df['RHOB_CLEAN'].isna()
affected_wells = df.loc[is_nan, 'well_name'].unique()
print(f"Remaining Massive Gaps affect {len(affected_wells)} wells out of {df['well_name'].nunique()} total wells.")
# missing rows per well
print(df[is_nan].groupby('well_name').size().sort_values(ascending=False))


In [ ]:
is_missing = df['RHOB_CLEAN'].isna()

# consecutive missing rows share a block id
blocks = (~is_missing).cumsum()

gap_sizes = df[is_missing].groupby(['well_name', blocks]).size()

avg_rows = gap_sizes.mean()

# logs are sampled every 0.1524 m (6 in)
avg_meters = avg_rows * 0.1524  

print(f"Total number of distinct gaps : {len(gap_sizes):,}")
print(f"Average gap size (in rows)    : {avg_rows:.1f} rows")
print(f"Average physical gap depth    : ~{avg_meters:.2f} meters")

max_rows = gap_sizes.max()
median_rows = gap_sizes.median()
min_rows = gap_sizes.min()

print(f"Maximum physical gap depth    : {max_rows} rows (~{max_rows * 0.1524:.2f} meters)")
print(f"Median physical gap depth    : {median_rows} rows (~{median_rows * 0.1524:.2f} meters)")
print(f"Minimum physical gap depth    : {min_rows} rows (~{min_rows * 0.1524:.2f} meters)")


In [ ]:
df

**Dealing with missing values:**

Option 1 : Train a simple XGBoost using GR, NPHI, DT to predict RHOB_CLEAN

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb

RMSE_THRESHOLD = 1
cols_to_impute = ['RHOB_CLEAN', 'GR_CLEAN', 'DS_INDEX', 'DEEPRES_CLEAN']

# checks before imputing
well_coverage = df.groupby('well_name')[cols_to_impute].count()
dead_wells = well_coverage[well_coverage.sum(axis=1) == 0].index
if len(dead_wells) > 0:
    print(f"🗑️  Dropping {len(dead_wells)} completely dead wells:")
    for w in dead_wells:
        print(f"    - {w}")
    df = df[~df['well_name'].isin(dead_wells)]
else:
    print("No completely dead wells found.")

print(f"\nTotal Rows Before Triage: {len(df):,}")
df = df.dropna(subset=cols_to_impute, how='all')
print(f"Total Rows After Triage : {len(df):,}")
print("-" * 55)

# imputation
wells_dropped_total = set()


In [ ]:

for target_col in cols_to_impute:
    predictor_cols = [c for c in cols_to_impute if c != target_col]
    train_mask   = df[target_col].notna()
    predict_mask = df[target_col].isna()
    X_valid   = df.loc[train_mask, predictor_cols]
    y_valid   = df.loc[train_mask, target_col]
    X_predict = df.loc[predict_mask, predictor_cols]

    if len(X_predict) > 0:
        X_train, X_test, y_train, y_test = train_test_split(
            X_valid, y_valid, test_size=0.2, random_state=42
        )
        model = xgb.XGBRegressor(n_estimators=100, max_depth=6, random_state=42, tree_method='hist')
        model.fit(X_train, y_train)
        rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
        r2   = r2_score(y_test, model.predict(X_test))

        print(f"\n🎯 Target: {target_col}")
        print(f"   RMSE     : {rmse:.4f}  |  R² : {r2:.4f} ({r2*100:.1f}%)")
        print(f"   Gap rows : {len(X_predict):,}")

        if rmse <= RMSE_THRESHOLD:
            df.loc[predict_mask, target_col] = model.predict(X_predict)
            print(f"   ✅ RMSE below threshold → Gaps filled.")
        else:
            guilty_wells = df.loc[predict_mask, 'well_name'].unique()
            wells_dropped_total.update(guilty_wells)
            df = df[~df['well_name'].isin(guilty_wells)]
            print(f"  Model too weak! Dropped {len(guilty_wells)} wells: {list(guilty_wells)}")
    else:                                                     
        print(f"\n Target: {target_col} → No gaps found. Skipped.")

print("\n" + "-" * 55)
print(f" Imputation Complete!")
print(f"   Total wells dropped (unresolvable) : {len(wells_dropped_total)}")
print(f"   Final well count                   : {df['well_name'].nunique()}")
print(f"   Final dataset size                 : {len(df):,} rows")
print(f"   Remaining NaNs in key logs         : {df[cols_to_impute].isna().sum().to_dict()}")

In [ ]:
df

In [ ]:
df.isna().sum()

In [ ]:
df

In [ ]:
import numpy as np
import pandas as pd
# tops
tops = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/north_sea/notebooks/Data_NPD/npd_tops.parquet')

tops = tops.reset_index()

# drop the coordinates so melt does not treat them as markers
tops = tops.drop(columns=['Latitude', 'Longitude'], errors='ignore')

tops_long = tops.melt(
    id_vars=['well_name'], 
    var_name='valid_marker_name', 
    value_name='md'
)

# markers a well did not reach are NaN
tops_filtered = tops_long.dropna(subset=['md'])



# sort logs by well and depth
df_sorted = (
    df.reset_index(drop=True)
    .dropna(subset=['DS_INDEX'])
    .assign(DS_INDEX=lambda x: pd.to_numeric(x['DS_INDEX'], errors='coerce'))
    .dropna(subset=['DS_INDEX'])
    .sort_values(['well_name', 'DS_INDEX'])
    .reset_index(drop=True)
)

print(f"df_sorted shape: {df_sorted.shape}")

# label each log row with searchsorted
labeled_dfs = []

for well, group in df_sorted.groupby('well_name'):
    group = group.copy()
    well_tops = tops_filtered[tops_filtered['well_name'] == well].sort_values('md')
    
    if well_tops.empty:
        group['marker_label'] = 'UNKNOWN'
        labeled_dfs.append(group)
        continue
    
    depths = well_tops['md'].values
    labels = well_tops['valid_marker_name'].values
    
    # formation interval each log row falls in
    idx = np.searchsorted(depths, group['DS_INDEX'].values, side='right') - 1
    
    group['marker_label'] = np.where(idx >= 0, labels[idx], 'UNKNOWN')
    labeled_dfs.append(group)

# df_sorted has a unique integer index, so the concat lines up
df = pd.concat(labeled_dfs, ignore_index=True)

print(f"\nFinal df shape      : {df.shape}")
print(f"Unique markers found: {df['marker_label'].nunique()}")
print(df['marker_label'].value_counts().head(10))


In [ ]:
# DEEPRES_CLEAN from natural log to log10
df['DEEPRES_CLEAN'] = np.exp(df['DEEPRES_CLEAN'])
df['DEEPRES_CLEAN'] = np.log10(df['DEEPRES_CLEAN'])

# min-max normalisation with fixed ranges

min_max_dict = {
    "GR_CLEAN":      [0.0, 200.0],
    "RHOB_CLEAN":    [1.95, 2.95],
    "DEEPRES_CLEAN": [np.log10(0.1), np.log10(2000.0)],
    "DS_INDEX":      [0.0, 10000.0]
}
# values outside the range are clipped to [0, 1]
for col, (vmin, vmax) in min_max_dict.items():
    if col in df.columns:
        df[col] = ((df[col] - vmin) / (vmax - vmin)).clip(0, 1)


In [ ]:
top_cols = ["DUNLIN", "HORDALAND", "ROGALAND", "SHETLAND", "VESTLAND", "VIKING"]

tops_count = df_tops[top_cols].notna().sum(axis=1)

# wells with at least 4 tops
wells_with_4_tops = set(tops_count[tops_count >= 4].index)

all_wells = [w for w in df['well_name'].unique() if w in wells_with_4_tops]

print(f"Filtered down to {len(all_wells)} wells with >= 4 tops out of {len(df['well_name'].unique())} total logs")


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

all_wells = df['well_name'].unique().tolist()
# 80/20 well split, no separate test set
train_wells, valid_wells = train_test_split(all_wells, test_size=0.2, random_state=42)


df_log_train = df[df['well_name'].isin(train_wells)].copy()
df_log_valid = df[df['well_name'].isin(valid_wells)].copy()

# try_run.py needs well_name as the index of the tops tables
# npd_tops.parquet is already one row per well
df_tops = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/north_sea/notebooks/Data_NPD/npd_tops.parquet')

df_tops_train = df_tops[df_tops.index.isin(train_wells)].copy()
df_tops_valid = df_tops[df_tops.index.isin(valid_wells)].copy()

# inputs for try_run.py
save_dir = "/Users/fokia/CRP_SLB/datasets/north_sea/Transformer/"

df_log_train.to_parquet(save_dir + 'log_train.parquet', engine='fastparquet')
df_log_valid.to_parquet(save_dir + 'log_valid.parquet', engine='fastparquet')
df_tops_train.to_parquet(save_dir + 'tops_train.parquet', engine='fastparquet')
df_tops_valid.to_parquet(save_dir + 'tops_valid.parquet', engine='fastparquet')

print(f"Saved {len(train_wells)} training wells and {len(valid_wells)} validation wells to {save_dir}")


In [ ]:
print("Index name  :", df.index.name)
print("Index sample:", df.index[:5].tolist())
print("Is unique   :", df.index.is_unique)
print("well_name is a column:", 'well_name' in df.columns)


In [ ]:
df_log_train

In [ ]:
tops_filtered

In [ ]:
df

In [ ]:
import sys, os
sys.path.insert(0, '/Users/fokia/CRP_SLB/datasets/colorado/Transformer')
os.chdir('/Users/fokia/CRP_SLB/datasets/colorado/Transformer')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from PatchTST_backbone import PatchTST_backbone

WSIZE     = 201
n_classes = tops_filtered['valid_marker_name'].nunique() + 1   # 5 tops + 1 background zone
n_markers_param = n_classes - 2  # PatchTST_backbone outputs n_markers + 2 classes
input_variables = ['GR_CLEAN', 'RHOB_CLEAN', 'DEEPRES_CLEAN']

print(f"n_classes       : {n_classes}")
print(f"n_markers_param : {n_markers_param}  (backbone will output {n_markers_param+2} logits)")


In [ ]:
import numpy as np
import pandas as pd
from tsai.all import *

def marker_signature_ns(df_wgr, marker, wsize):
    hd = int(wsize * 0.5)
    std = 2
    md = marker
    dseq = list(range(int(md - (hd + std)), int(md + (hd + std))))
    # log rows within the window around the marker
    seq = df_wgr[df_wgr.index.isin(dseq)].to_numpy()
    
    mseg = []
    for i in range(len(seq) - wsize + 1):
        mseg.append(np.transpose(seq[i: i + wsize]))
    return np.array(mseg)
def extract_dataset_Xy_ns(logs, tops, well_list, input_variable, wsize, top_list_bool=0):
    X = np.empty(shape=(0, len(input_variable), wsize))
    y = np.empty(shape=(0))

    for well in well_list:
        # skip wells missing from the logs or the tops table
        if well not in logs['well_name'].unique() or well not in tops.index:
            continue
        
        df_wgr = logs[logs['well_name'] == well].copy()
        df_wgr['DEPTH_INT'] = df_wgr['DS_INDEX'].astype(int)
        df_wgr = df_wgr.groupby('DEPTH_INT').mean(numeric_only=True)
        df_wgr = df_wgr[input_variable]
        
        for i in range(len(tops.columns)):
            marker_depth = tops.loc[well].iloc[i]
            if pd.isna(marker_depth): continue
            
            np_mseg = marker_signature_ns(df_wgr, marker_depth, wsize)
            y_seq = np.full(len(np_mseg), i + 1)
            
            if len(np_mseg) > 0:
                X = np.append(X, np_mseg, axis=0)
                y = np.append(y, y_seq, axis=0)

        wtop_valid = tops.loc[well].dropna().values
        if len(wtop_valid) >= 2:
            low_d  = int(np.nanmin(wtop_valid))
            high_d = int(np.nanmax(wtop_valid))
            if high_d > low_d:
                rd = np.random.randint(low=low_d, high=high_d)
                np_mseg_bg = marker_signature_ns(df_wgr, rd, wsize)
                if len(np_mseg_bg) > 0:
                    X = np.append(X, np_mseg_bg[:1], axis=0)
                    y = np.append(y, [0], axis=0)
            
    return X, y



In [ ]:
print("Unique well names in tops:")
print(tops['well_name'].unique())

In [ ]:
from sklearn.model_selection import train_test_split

# wells with the 5 selected markers
all_wells = tops['well_name'].unique().tolist()

# 80/20 well split
train_wells, valid_wells = train_test_split(all_wells, test_size=0.2, random_state=42)

print(f"✅ Well-level split complete:")
print(f"   - Total wells with markers: {len(all_wells)}")
print(f"   - Wells for Training     : {len(train_wells)}")
print(f"   - Wells for Validation   : {len(valid_wells)}")

assert len(set(train_wells).intersection(set(valid_wells))) == 0


In [ ]:
df['marker_label'].value_counts()

In [ ]:
tops_wide

In [ ]:
input_variable = ['GR_CLEAN', 'RHOB_CLEAN', 'DEEPRES_CLEAN']
WSIZE = 201

# train_wells and valid_wells come from the split cell above
X_train, y_train = extract_dataset_Xy_ns(df, tops_wide,train_wells, input_variable, WSIZE)
X_valid, y_valid = extract_dataset_Xy_ns(df, tops_wide,valid_wells, input_variable, WSIZE)

X, y, splits = combine_split_data([X_train, X_valid], [y_train, y_valid])

tfms  = [None, [Categorize()]]
dsets = TSDatasets(X, y, tfms=tfms, splits=splits, inplace=True)

dls = TSDataLoaders.from_dsets(
    dsets.train, 
    dsets.valid, 
    bs=25, 
    batch_tfms=[TSNormalize(by_var=True, range=(0,1))], 
    num_workers=0
)

print(f"Dataset created with {len(dls.train)} training batches.")
dls.show_batch()


In [ ]:
model = PatchTST_backbone(
    c_in           = len(input_variables),  # 3
    target_window  = 1,                     # one prediction per window
    context_window = WSIZE,                 # 201
    patch_len      = 16,
    stride         = 8,
    n_markers      = n_markers_param,       # n_classes - 2
)

In [ ]:
X_train

In [ ]:
X_tr = torch.tensor(X_train, dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.long)
X_va = torch.tensor(X_valid, dtype=torch.float32)
y_va = torch.tensor(y_valid, dtype=torch.long)

train_dl = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
valid_dl = DataLoader(TensorDataset(X_va, y_va), batch_size=32)


In [ ]:

device    = torch.device("cpu")   # use "cuda" on a GPU machine
model     = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

for epoch in range(1, 21):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X_b, y_b in train_dl:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        out   = model(X_b).squeeze(1)          # (bs, 1, n_classes) to (bs, n_classes)
        loss  = F.cross_entropy(out, y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_b)
        correct    += (out.argmax(1) == y_b).sum().item()
        total      += len(y_b)
    scheduler.step()

    # validation
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for X_b, y_b in valid_dl:
            X_b, y_b = X_b.to(device), y_b.to(device)
            out       = model(X_b).squeeze(1)
            val_loss += F.cross_entropy(out, y_b).item() * len(y_b)
            val_correct += (out.argmax(1) == y_b).sum().item()
            val_total   += len(y_b)

    print(f"Epoch {epoch:2d}/20 | "
          f"train_loss={total_loss/total:.4f}  train_acc={correct/total:.4f} | "
          f"val_loss={val_loss/val_total:.4f}  val_acc={val_correct/val_total:.4f}")


In [ ]:
import sys, os
sys.path.insert(0, '/Users/fokia/CRP_SLB/datasets/colorado/Transformer')
os.chdir('/Users/fokia/CRP_SLB/datasets/colorado/Transformer')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from typing import List, Optional, Tuple

# reuses the Colorado transformer code
from PatchTST_backbone import PatchTST_backbone
from train import (
    WellLogDataset, SegmentationHead,
    build_tops_seq, crop_to_tops_region,
    train, predict,
    evaluate_top_picking, evaluate_segmentation_metrics, plot_prediction
)
print("✅ Colorado Transformer imported successfully")


In [ ]:
# tops to one row per well, same layout as Colorado
# needs tops_filtered from the cells above

# markers present in the most wells
marker_well_count = tops_filtered.groupby('valid_marker_name')['well_name'].nunique()
top_markers = marker_well_count.sort_values(ascending=False).head(5).index.tolist()
print(f"Top 5 markers by well coverage: {top_markers}")

# one row per well, one column per marker, values are md
tops_wide = (
    tops_filtered[tops_filtered['valid_marker_name'].isin(top_markers)]
    .pivot_table(index='well_name', columns='valid_marker_name', values='md', aggfunc='first')
    [top_markers]
)

# keep wells with all 5 markers, partly labelled wells would give wrong zone labels
tops_wide = tops_wide.dropna()
print(f"\nWells with complete marker coverage : {len(tops_wide)}")
print(tops_wide.head())


In [ ]:
# Colorado's prepare_sequences with 3 log channels
input_variables = ['GR_CLEAN', 'RHOB_CLEAN', 'DEEPRES_CLEAN']
depth_col = 'DS_INDEX'
well_col  = 'well_name'

def prepare_sequences_north_sea(
    df_log:       pd.DataFrame,
    df_tops:      pd.DataFrame,   # indexed by well_name, one column per marker
    top_cols:     List[str],
    feature_cols: List[str],
    depth_col:    str   = 'DS_INDEX',
    well_col:     str   = 'well_name',
    srate:        float = 4.0,      # depth step of the resampled grid
    margin_ft:    float = 500.0,
    pad_value:    float = 0.0,
    well_list:    Optional[List] = None,
) -> Tuple[np.ndarray, np.ndarray, List[np.ndarray]]:
    """
    Returns:
        X          (n_wells, n_features, seq_len)   float32
        y          (n_wells, seq_len)               int64 zone labels
        depth_axes list of unpadded depth arrays
    """
    log_wells  = set(df_log[well_col].unique())
    tops_wells = set(df_tops.index)
    candidates = well_list if (well_list and len(well_list) > 0) else list(tops_wells)

    wells = [w for w in candidates
             if w in log_wells
             and w in tops_wells
             and not df_tops.loc[w, top_cols].isnull().any()]

    all_features, all_labels, depth_axes = [], [], []

    for well in wells:
        wdf = df_log[df_log[well_col] == well].sort_values(depth_col)
        depth_raw = wdf[depth_col].values.astype(np.float32)

        min_d = depth_raw.min()
        max_d = depth_raw.max()
        depth_grid = np.arange(min_d, max_d + srate, srate, dtype=np.float32)

        feats_resampled = []
        for feat in feature_cols:
            raw = wdf[feat].ffill().bfill().values.astype(np.float32)
            resampled = np.interp(depth_grid, depth_raw, raw).astype(np.float32)
            feats_resampled.append(resampled)
        feats_arr = np.stack(feats_resampled, axis=0)   # (n_feats, n_depth)

        # crop to the true tops ± margin
        top_depths = df_tops.loc[well, top_cols].values.astype(float)
        min_top = top_depths.min() - margin_ft
        max_top = top_depths.max() + margin_ft
        mask = (depth_grid >= min_top) & (depth_grid <= max_top)

        depth_grid   = depth_grid[mask]
        feats_arr    = feats_arr[:, mask]
        labels       = build_tops_seq(depth_grid, top_depths)

        all_features.append(feats_arr)
        all_labels.append(labels)
        depth_axes.append(depth_grid)

    if len(all_features) == 0:
        raise ValueError("No valid wells found! Check well name matching between df_log and df_tops.")

    seq_len = max(f.shape[1] for f in all_features)
    n_feats = len(feature_cols)
    n_wells = len(all_features)

    X = np.full((n_wells, n_feats, seq_len), pad_value, dtype=np.float32)
    y = np.zeros((n_wells, seq_len), dtype=np.int64)

    for i, (feats, lbl) in enumerate(zip(all_features, all_labels)):
        L = feats.shape[1]
        X[i, :, :L] = feats
        y[i, :len(lbl)] = lbl
        y[i, len(lbl):] = lbl[-1]   # padding gets the last zone label

    print(f"Prepared {n_wells} wells | seq_len={seq_len} | "
          f"n_classes={len(top_cols)+1} | n_features={n_feats}")
    return X, y, depth_axes


In [ ]:
# PatchTSTSegmenter from train.py with c_in as a parameter instead of 1
class NorthSeaSegmenter(nn.Module):
    def __init__(
        self,
        c_in:         int,           # 3 for GR_CLEAN, RHOB_CLEAN, DEEPRES_CLEAN
        seq_len:      int,
        n_classes:    int,
        patch_len:    int   = 16,
        stride:       int   = 8,
        d_model:      int   = 128,
        n_heads:      int   = 8,
        n_layers:     int   = 3,
        d_ff:         int   = 256,
        n_markers:    int   = 0,
        dropout:      float = 0.1,
        head_dropout: float = 0.1,
    ):
        super().__init__()
        patch_num = int((seq_len - patch_len) / stride + 1) + 1  # +1 for 'end' padding
        head_nf   = (n_markers + 2) * patch_num

        self.backbone = PatchTST_backbone(
            c_in           = c_in,
            context_window = seq_len,
            target_window  = seq_len,     # placeholder
            patch_len      = patch_len,
            n_markers      = n_markers,
            stride         = stride,
            d_model        = d_model,
            n_heads        = n_heads,
            n_layers       = n_layers,
            d_ff           = d_ff,
            norm           = "BatchNorm",
            dropout        = dropout,
            fc_dropout     = dropout,
            head_dropout   = head_dropout,
            head_type      = "flatten",
            individual     = False,
            revin          = False,
            padding_patch  = "end",
            pe             = "zeros",
            learn_pe       = True,
            res_attention  = True,
        )
        del self.backbone.head   # replaced by SegmentationHead

        self.seg_head = SegmentationHead(
            head_nf   = head_nf,
            n_vars    = c_in,
            seq_len   = seq_len,
            n_classes = n_classes,
            dropout   = head_dropout,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self._encode(x)
        return self.seg_head(z)

    def _encode(self, z: torch.Tensor) -> torch.Tensor:
        b = self.backbone
        if b.padding_patch == "end":
            z = b.padding_patch_layer(z)
        z = z.unfold(dimension=-1, size=b.patch_len, step=b.stride)
        z = z.permute(0, 1, 3, 2)
        kpm = None
        if b.key_padding_mask == "auto":
            z, kpm = b.key_padding_masking(z)
        z = torch.nan_to_num(z)
        z = b.backbone(z, key_padding_mask_tensor=kpm)
        return z


In [ ]:
N_WELLS_DEBUG = 20
all_wells = all_wells[:N_WELLS_DEBUG]
print(f"DEBUG MODE: Running on {len(all_wells)} wells only")
train_wells, valid_wells = train_test_split(all_wells, test_size=0.2, random_state=42)
# 16 train, 4 valid

In [ ]:
# well-level train/valid split
top_cols   = top_markers
n_classes  = len(top_cols) + 1

all_wells = tops_wide.index.tolist()
train_wells, valid_wells = train_test_split(all_wells, test_size=0.2, random_state=42)

print(f"Train wells : {len(train_wells)}")
print(f"Valid wells : {len(valid_wells)}")

df_train_log = df[df[well_col].isin(train_wells)]
df_valid_log = df[df[well_col].isin(valid_wells)]

print("\nPreparing training sequences …")
X_train, y_train, depth_axes_train = prepare_sequences_north_sea(
    df_train_log, tops_wide, top_cols, input_variables,
    depth_col=depth_col, well_col=well_col, well_list=train_wells
)

print("\nPreparing validation sequences …")
X_valid, y_valid, depth_axes_valid = prepare_sequences_north_sea(
    df_valid_log, tops_wide, top_cols, input_variables,
    depth_col=depth_col, well_col=well_col, well_list=valid_wells
)

# pad both splits to the same seq_len
seq_len = max(X_train.shape[2], X_valid.shape[2])

def pad_to(X, y, L):
    if X.shape[2] < L:
        pad_X = np.zeros((len(X), X.shape[1], L - X.shape[2]), dtype=X.dtype)
        pad_y = np.full((len(y), L - y.shape[1]), y[:, -1:], dtype=y.dtype)
        X = np.concatenate([X, pad_X], axis=2)
        y = np.concatenate([y, pad_y], axis=1)
    return X, y

X_train, y_train = pad_to(X_train, y_train, seq_len)
X_valid, y_valid = pad_to(X_valid, y_valid, seq_len)

print(f"\nX_train : {X_train.shape}  |  y_train : {y_train.shape}")
print(f"X_valid  : {X_valid.shape}   |  y_valid  : {y_valid.shape}")
print(f"n_classes: {n_classes}  |  seq_len: {seq_len}")


In [ ]:
model = NorthSeaSegmenter(
    c_in      = len(input_variables),   # 3
    seq_len   = seq_len,
    n_classes = n_classes,
    patch_len = 16,
    stride    = 8,
    d_model   = 128,
    n_heads   = 8,
    n_layers  = 3,
    d_ff      = 256,
    n_markers = 0,
    dropout   = 0.1,
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

# Colorado train() loop
model = train(
    model, X_train, y_train, X_valid, y_valid,
    epochs     = 30,
    batch_size = 16,
    lr         = 1e-4,
    save_path  = '../../north_sea/checkpoints/patchtst_north_sea.pt',
)


In [ ]:
# interpolation within each well
df['RHOB_CLEAN'] = df.groupby('well_name')['RHOB_CLEAN'].transform(lambda group: group.interpolate(method='linear'))


In [ ]:
import pandas as pd 
!pip install fastparquet
# GR downsampled to one value per foot
df_log = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/colorado/data/data/Training/logs.parquet', engine='fastparquet')
df_log.loc[df_log['GR'] < -1, 'GR' ] = -1
df_loc = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/colorado/data/data/Training/loc.parquet', engine='fastparquet')
df_loc = df_loc.reset_index()
df_log = df_loc.merge(df_log, how = 'inner', left_on = 'wellName', right_on = 'wellName')
print(df_log.head())

In [ ]:
# 5 markers present in the most wells
top_5_markers = (
    tops.groupby('valid_marker_name')['well_name']
    .nunique()
    .sort_values(ascending=False)
    .head(5)
    .index.tolist()
)
print(f"✅ Top 5 Target Markers: {top_5_markers}")

# one row per well
tops_wide = (
    tops[tops['valid_marker_name'].isin(top_5_markers)]
    .pivot_table(
        index='well_name', 
        columns='valid_marker_name', 
        values='md',
        aggfunc='first'
    )
)

# same column order as top_5
tops_wide = tops_wide[top_5_markers]

print("\nSuccess! This matches your target format:")
display(tops_wide.head())


In [ ]:
tops

In [ ]:
import numpy as np
# training wells, the ones with a good signature cluster
well_array = np.load('/Users/fokia/CRP_SLB/datasets/colorado/data/data/Training/training_well_list.npy', allow_pickle=True)

In [ ]:
from geoxts.benchmark_data import *

In [ ]:
input_variable = ['GR', 'Depth','Latitude','Longitude']
# training data
X_train, y_train = extract_dataset_Xy(df_log,df_tops,well_array, input_variable, wsize=201, top_list_bool=1 )
X_train.shape, y_train.shape

In [ ]:
df_valid_log

In [ ]:
# validation set for the recall calculation
df_valid_log = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/colorado/data/data/test/logs_50.parquet', engine='fastparquet')
df_valid_log.loc[df_valid_log['GR'] < -1, 'GR' ] = -1
df_valid_log.loc[df_valid_log['GR'] > 400, 'GR' ] = 400
df_valid_loc = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/colorado/data/data/test/loc_50.parquet', engine='fastparquet')
df_valid_loc = df_valid_loc.reset_index()
df_valid_log = df_valid_loc.merge(df_valid_log, how = 'inner', left_on = 'wellName', right_on = 'wellName')
df_valid_tops = pd.read_csv('/Users/fokia/CRP_SLB/datasets/colorado/data/data/test/tops_50.csv')
df_valid_tops = df_valid_tops.set_index('wellName')
cols = ['NIOBARA','CODELL', 'FORT_HAYS']
df_valid_tops = df_valid_tops[cols]

In [ ]:
X_valid, y_valid = extract_dataset_Xy(df_valid_log,df_valid_tops,[],input_variable, wsize=201, top_list_bool=0)
X_valid.shape, y_valid.shape

In [ ]:
from tsai.all import *


In [ ]:
X, y, splits = combine_split_data([X_train, X_valid], [y_train, y_valid])
tfms  = [None, [Categorize()]]
dsets = TSDatasets(X, y, tfms=tfms, splits=splits, inplace=True)
dls = TSDataLoaders.from_dsets(dsets.train, dsets.valid, bs=25, batch_tfms=[TSNormalize(by_var=True, range = (0,1))], num_workers=0) 

In [ ]:
dsets

In [ ]:
valid_dl = dls.valid
dls.show_batch()

In [ ]:
# test set for the recall calculation
df_test_log = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/colorado/data/data/test/logs_100.parquet', engine='fastparquet')
df_test_log.loc[df_test_log['GR'] < -1, 'GR' ] = -1
df_test_log.loc[df_test_log['GR'] > 400, 'GR' ] = 400
df_test_loc = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/colorado/data/data/test/loc_100.parquet', engine='fastparquet')
df_test_loc = df_test_loc.reset_index()
df_test_log = df_test_loc.merge(df_test_log, how = 'inner', left_on = 'wellName', right_on = 'wellName')
df_test_tops = pd.read_csv('/Users/fokia/CRP_SLB/datasets/colorado/data/data/test/tops_100.csv')
df_test_tops = df_test_tops.set_index('wellName')
cols = ['NIOBARA','CODELL', 'FORT_HAYS']
df_test_tops = df_test_tops[cols]

In [ ]:
batch_size = 64
dls = TSDataLoaders.from_dsets(dsets.train, dsets.valid, bs=batch_size)


In [ ]:
import sys
sys.path.append('/Users/fokia/CRP_SLB/datasets/colorado/Transformer')

from PatchTST_backbone import PatchTST_backbone 


In [ ]:
model = PatchTST_backbone(
    c_in=X.shape[1],
    target_window=1,
    context_window=X.shape[2],
    patch_len=16,
    stride=8,
    n_markers=4
)


In [ ]:
%reload_ext autoreload
%autoreload 2 


In [ ]:
df_log

In [ ]:

learn = Learner(dls, model, metrics=accuracy)

# 10 epochs, max lr 1e-3
learn.fit_one_cycle(10, lr_max=1e-3)

# loss curves:
# learn.recorder.plot_loss()


In [ ]:
learn.recorder.plot_loss()


In [ ]:
learn.show_results()


In [ ]:
# marker name to class index
print(dls.vocab)


In [ ]:
import numpy as np

# predictions and targets on the validation set
probs, targets = learn.get_preds(ds_idx=1)

# first window, change idx to look at others
idx = 0  

# one class per window, repeated over its 200 points for plotting
predicted_class = probs[idx].argmax().item()
predicted_zones = np.full(200, predicted_class)

# channel 0 of the window is GR
gr_log = X_valid[idx][0]
seq_len = len(gr_log)

predicted_zones = np.full(seq_len, predicted_class) 
depths = np.arange(seq_len) 


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6), sharey=True)

# left: input GR
ax1.plot(gr_log, depths, color='steelblue', linewidth=1)
ax1.set_title(f'GR Log | Patch {idx}')
ax1.set_ylabel('Depth Steps')
ax1.invert_yaxis()  # depth increases downwards

# right: prediction
ax2.plot(predicted_zones, depths, color='steelblue', drawstyle='steps-pre')
ax2.set_title(f'Model Prediction\n(Guessed: Class {predicted_class} | True: {targets[idx].item()})')
ax2.set_xticks([0, 1, 2, 3, 4, 5]) # one tick per class

plt.tight_layout()
plt.show()


In [ ]:
X_well[:, 1, 100]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# one well, fixed seed
np.random.seed(42)  
well_id = df_valid_tops.index[10]  

X_well, y_well = extract_dataset_Xy(
    df_valid_log, df_valid_tops, [well_id], input_variable, 
    wsize=201, top_list_bool=0
)

probs, targets, _ = learn.get_X_preds(X_well, y_well)
preds = probs.argmax(dim=-1).numpy().flatten()

# depth of each prediction
# channel 1 of X_well is depth, index 100 is the centre of the window
actual_depths = X_well[:, 1, 100]

# sort by depth for the step plot
sort_idx = np.argsort(actual_depths)
actual_depths = actual_depths[sort_idx]
preds = preds[sort_idx]

# full-resolution GR from the dataframe
well_data = df_valid_log[df_valid_log['wellName'] == well_id]
raw_depths = well_data['DEPTH'].values
raw_gr = well_data['GR'].values


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 8), sharey=True)

# left: GR log
ax1.plot(raw_gr, raw_depths, color='#5c85b1', alpha=0.9, linewidth=1)
ax1.set_title(f'GR Log | {well_id}')
ax1.set_xlabel('GR (API)')
ax1.set_ylabel('Depth (ft)')

# depth increases downwards
ax1.invert_yaxis()  

# right: predicted zones
ax2.plot(preds, actual_depths, color='#5c85b1', linewidth=2, drawstyle='steps-pre')
ax2.set_title('Zone step function')
ax2.set_xlabel('Predicted zone')
ax2.set_xticks([0, 1, 2, 3])

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# one well, fixed seed
np.random.seed(42)  
well_id = df_valid_tops.index[0]  

X_well, y_well = extract_dataset_Xy(
    df_valid_log, df_valid_tops, [well_id], input_variable, 
    wsize=201, top_list_bool=0
)

probs, targets, _ = learn.get_X_preds(X_well, y_well)
preds = probs.argmax(dim=-1).numpy().flatten()

# depth of each prediction
# channel 1 of X_well is depth, index 100 is the centre of the window
actual_depths = X_well[:, 1, 100]

# sort by depth for the step plot
sort_idx = np.argsort(actual_depths)
actual_depths = actual_depths[sort_idx]
preds = preds[sort_idx]

# full-resolution GR from the dataframe
well_data = df_valid_log[df_valid_log['wellName'] == well_id]
raw_depths = well_data['DEPTH'].values
raw_gr = well_data['GR'].values


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 8), sharey=True)

# left: GR log
ax1.plot(raw_gr, raw_depths, color='#5c85b1', alpha=0.9, linewidth=1)
ax1.set_title(f'GR Log | {well_id}')
ax1.set_xlabel('GR (API)')
ax1.set_ylabel('Depth (ft)')

# depth increases downwards
ax1.invert_yaxis()  

# right: predicted zones
ax2.plot(preds, actual_depths, color='#5c85b1', linewidth=2, drawstyle='steps-pre')
ax2.set_title('Zone step function')
ax2.set_xlabel('Predicted zone')
ax2.set_xticks([0, 1, 2, 3])

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

formation_names = ['NIOBARA', 'CODELL', 'FORT_HAYS']
top_names = ['zone_0'] + formation_names

# accuracy and classification metrics
probs, targets = learn.get_preds(ds_idx=1)
preds = probs.argmax(dim=-1).numpy().flatten()
targets = targets.numpy().flatten()

print(f"Overall depth-sample accuracy: {(preds == targets).mean():.4f}\n")
print("Per-class metrics:")
print(classification_report(targets, preds, target_names=top_names, digits=2))

print("Confusion matrix (rows=true, cols=pred):")
cm = confusion_matrix(targets, preds)
print(pd.DataFrame(cm, index=top_names, columns=top_names))
print("\n" + "="*50 + "\n")

# top-picking MAE
mae_errors = {name: [] for name in formation_names}
print("Top-picking mean absolute error:")

for well_id in df_valid_tops.index:
    true_tops = df_valid_tops.loc[well_id] 
    
    X_well, y_well = extract_dataset_Xy(
        df_valid_log, df_valid_tops, [well_id], input_variable, 
        wsize=201, top_list_bool=0
    )
    if len(X_well) == 0: continue
    
    depths = X_well[:, 1, 100] 
    probs, _, _ = learn.get_X_preds(X_well, y_well)
    well_preds = probs.argmax(dim=-1).numpy().flatten()
    
    sort_idx = np.argsort(depths)
    depths = depths[sort_idx]
    well_preds = well_preds[sort_idx]

    
    for zone_idx, formation in enumerate(formation_names, start=1):
        true_depth = true_tops[formation]
        pred_idx = np.where(well_preds == zone_idx)[0]
        
        if len(pred_idx) > 0 and not pd.isna(true_depth):
            predicted_depth = depths[pred_idx[0]]
            mae_errors[formation].append(abs(predicted_depth - true_depth))

for formation in formation_names:
    if len(mae_errors[formation]) > 0:
        print(f"  {formation:<12} {np.mean(mae_errors[formation]):>5.1f} ft")


In [ ]:
print("Accuracy per marker (True Positive Rate):")
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

for name, acc in zip(top_names, per_class_accuracy):
    print(f"  {name:<12} {acc*100:>5.1f}%")


In [ ]:
print("Accuracy per marker (True Positive Rate):")
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

for name, acc in zip(top_names, per_class_accuracy):
    print(f"  {name:<12} {acc*100:>5.1f}%")